In [1]:
!pip -q install "transformers>=4.42.0" "accelerate>=0.31.0" "bitsandbytes>=0.43.0" \
                "langchain>=0.2.0" "langchain-huggingface>=0.0.3" \
                "datasets>=2.19.0" "tqdm" "pandas>=2.0.0"


## 데이터 불러오기

In [2]:
import json, pandas as pd, requests

RAW_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/dev.json"
data = requests.get(RAW_URL).json()
df = pd.DataFrame(data)

## 모델 불러오기

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # GPU 메모리 충분 → bf16 권장
    device_map="auto",            # Accelerate가 최적 배치(멀티 GPU 포함)
)
model.eval()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

## 시스템 프롬프트 & 대화 템플릿 설정

In [7]:
SYSTEM_PROMPT = (
    "당신은 대화 맥락 추론 전문가입니다. "
    "아래 대화와 보기(A/B/C)가 주어지면, 가장 가능성 높은 답을 "
    "대문자 하나(A/B/C)만 출력하세요. 해설/추가텍스트 금지."
)

def build_messages(dialogue: str, question: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"[대화]\n{dialogue}\n\n[문항]\n{question}\n\n출력 형식: A 또는 B 또는 C"}
    ]

# Llama 3 系열은 <|eot_id|>를 turn 종료로 쓰기도 하므로(eos 외 추가 종료토큰)
# 존재하면 함께 등록 (문서: eos_token_id에 리스트 전달 가능)
terminators = [tokenizer.eos_token_id]
try:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id >= 0:
        terminators.append(eot_id)
except Exception:
    pass

## 예시 데이터로 모델 테스트 해보기

In [9]:
sample = df.iloc[0].to_dict()
messages = build_messages(sample["dialogue"], sample["question"])

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=16,       # 답은 한 글자 → 작게 설정
        eos_token_id=terminators,
        do_sample=False,         # 정확도 평가 → 샘플링 끔
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
print("[대화]\n", sample["dialogue"])
print("\n\n[질문]\n", sample["question"])
print("\n\n[모델 대답]:", generated)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[대화]
 화자1: 안녕하세요! ㅎㅎ
화자2: 안녕하세요!
화자2: 동호회활동하세요?
화자1: 저는 차 마시는 거 좋아해서 차 좋아하는 사람들이랑 가끔 다회 참석해요
화자1: name2님은 동호회활동 참여하는 거 있으신가요?
화자2: 오오
화자2: 아니요ㅋㅋㅋ저는 없어요
화자2: 새로운 사람 만나는 걸 즐기지는 않아서 아직 시도를 못 해봤어요 동호회!
화자1: 아하ㅎㅎ  저는 직업상 만나는 사람들이 한정적이어서 동호회 활동을 하니까 다양한 사람을 만나게 되어 좋더라고요
화자1: 저도 굉장히 내향적인 편인데 동호회 안에서 친해진 사람들하고만 만나고 있어요
화자2: 이상하게 동호회는 목적이 다르다는 편견이 좀 있어서 망설여지더라고요!
화자2: 물론 다 그런 게 아닌데!
화자1: 아 무슨 말씀이신지 너무 알 것 같아요
화자1: 불순한 목적으로 동호회 참석하는 사람들 때문에 좀 이미지가 그렇긴 하죠
화자1: 특히 남자..들.. 이런 말 해도 되나요 ㅋㅋㅋ
화자2: 으으 맞아요 맞아
화자2: 안 좋은 얘기도 사실 많이 들어서...
화자1: 제가 참석하는 차 동호회는 20대 여자가 대부분이라 더 편하게 만날 수 있는 것 같아요ㅠ
화자1: 여성 동호회 쪽으로 알아보시면 좀 걱정이 덜하지 않을까요
화자2: 오! 그런 방법이 있겠네요
화자2: 그리고 어떤 분이 그러셨는데
화자2: 약간 돈이 드는 동호회? 는 오히려 그런 사람이 적대요ㅋㅋㅋㅋㅋㅋ
화자1: 돈이 드는 동호회 ㅋㅋㅋㅋㅋ  차는 돈이 좀 들어서 그런 걸까요
화자2: 네! 그럴지도 몰라요
화자2: 동호회는 인터넷에서 검색해서 들어가는 건가요?
화자1: 저는 트위터로 시작해서 마음 맞는 사람들이랑 같이 만나고 있어요
화자2: 오호 트위터!
화자2: 그런 방법도 있네요 정말
화자2: 트위터하면 아무래도 관심사가 비슷한 사람을 만나게 되니까요
화자1: 맞아요 맞아요


[질문]
 위 대화 이후 일어날 가능성이 가장 높은 ‘다음 사건’을 선택하세요.
A. 화자2는 미혼 남녀 대상의 동호회를 알아볼 것이다. 
B. 화자2는 가

## 정답 파서 (모델 대답에서 A/B/C 만 추출)

In [12]:
import re

def extract_choice(text: str) -> str:
    if not text:
        return ""
    m = re.search(r"\b([ABC])\b", text.strip())
    if m: return m.group(1)
    m = re.search(r"[정답답안]\s*[:：]\s*([ABC])", text)
    if m: return m.group(1)
    m = re.search(r"[（(]\s*([ABC])\s*[)）]", text)
    if m: return m.group(1)
    # 마지막 방어막: 포함된 첫 A/B/C
    for c in "ABC":
        if c in text: return c
    return ""


## 전체 평가 코드

In [13]:
from tqdm import tqdm
import pandas as pd
import numpy as np

# 카테고리 컬럼명 탐색 (없으면 None)
possible_cols = ["category", "카테고리", "type", "question_category"]
category_col = next((c for c in possible_cols if c in df.columns), None)

preds, gts, ids, cats, raws = [], [], [], [], []

for row in tqdm(df.to_dict(orient="records")):
    # 1) 메시지 구성 & 템플릿 적용
    messages = build_messages(row["dialogue"], row["question"])
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    # 2) 생성
    with torch.no_grad():
        out_ids = model.generate(
            input_ids,
            max_new_tokens=16,
            eos_token_id=terminators,   # 여러 종료 토큰 허용
            do_sample=False,            # 평가 시 결정론적
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 3) 후처리(정답 파싱)
    gen_text = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
    pred = extract_choice(gen_text)
    gold = row["answer"].strip()

    # 4) 카테고리 가져오기 (없으면 UNKNOWN)
    if category_col is not None:
        cat = row.get(category_col, "UNKNOWN")
    else:
        cat = "UNKNOWN"

    preds.append(pred)
    gts.append(gold)
    ids.append(row["id"])
    cats.append(cat)
    raws.append(gen_text)


  3%|▎         | 4/151 [00:00<00:16,  9.06it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  5%|▌         | 8/151 [00:00<00:15,  9.04it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  7%|▋         | 11/151 [00:01<00:14,  9.55it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 결과 집계

In [14]:
# 결과 프레임
res = pd.DataFrame({
    "id": ids,
    "category": cats,
    "gold": gts,
    "pred": preds,
    "correct": [int(p == g) for p, g in zip(preds, gts)],
    "raw": raws,  # 원문 응답 확인용
})

# 카테고리별 집계
cat_order = ["후행사건", "동기", "전제", "반응"]
grp = (
    res.groupby("category", dropna=False)
       .agg(n=("correct", "size"), correct=("correct", "sum"))
       .assign(accuracy=lambda d: d["correct"] / d["n"])
)

# 보기 좋은 순서로 재정렬 (데이터에 존재하지 않으면 자동으로 스킵)
ordered_index = [c for c in cat_order if c in grp.index] + [c for c in grp.index if c not in cat_order]
grp = grp.loc[ordered_index]

# 전체 집계
overall = pd.DataFrame({
    "n": [int(res.shape[0])],
    "correct": [int(res["correct"].sum())],
    "accuracy": [res["correct"].mean() if res.shape[0] else np.nan],
}, index=["전체"])

# 최종 표
summary_table = pd.concat([grp, overall], axis=0)

# 퍼센트 포맷 버전(표시용)
display_table = summary_table.copy()
display_table["accuracy"] = (display_table["accuracy"] * 100).round(2).astype(str) + "%"

print("=== 카테고리별 및 전체 정확도 ===")
display(display_table)

# 텍스트로도 한 줄 출력
correct = int(res["correct"].sum())
total = int(res.shape[0])
print(f"\nOverall Accuracy: {correct}/{total} = {correct/total:.4f}")

=== 카테고리별 및 전체 정확도 ===


,n,correct,accuracy
후행사건,51,34,66.67%
동기,25,20,80.0%
전제,25,21,84.0%
반응,25,15,60.0%
원인,25,16,64.0%
전체,151,106,70.2%



Overall Accuracy: 106/151 = 0.7020
